In [ ]:
import pathlib

In [ ]:
import downstream
import imageio
import marimo as mo
from matplotlib import pyplot as plt
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import polars as pl
import seaborn as sns
from teeplot import teeplot as tp
from tqdm.auto import tqdm
from watermark import watermark

from pylib.munge import addend_groups, squeeze_int

In [ ]:
mo.md(
    f"""
```Text
{watermark(
    current_date=True,
    iso8601=True,
    machine=True,
    updated=True,
    python=True,
    iversions=True,
    globals_=globals(),
)}
```
"""
)

```Text
Last updated: 2026-11-21T03:10:32.447558+00:00

Python implementation: CPython
Python version       : 3.10.12
IPython version      : 7.31.1

Compiler    : GCC 11.4.0
OS          : Linux
Release     : 6.8.0-1070-azure
Machine     : x86_64
Processor   : x86_64
CPU cores   : 4
Architecture: 64bit

marimo    : 0.23.2
matplotlib: 3.10.7
numpy     : 2.1.2
PIL       : 12.2.0
seaborn   : 0.13.2
teeplot   : 1.4.2
polars    : 1.40.1
downstream: 1.25.0
imageio   : 2.37.0

```

# 2026-11-21-alleleevoepi-countdown-p2p-traitlog

Strains of each PE's sampled host (0th population member) over time,
from the trait logger of wse-async-ga workflow
`2026-11-21-alleleevoepi-countdown-p2p-traitlog` (genome
`genome_alleleevoepi_2026_11_21`, 2 sites), nested subgrid runs
sweeping mutation rate.

Genome `genome_alleleevoepi_2026_11_21` counts infection and immunity
durations down on nibbles, instead of being memoryless: infections last
`evoepi_infection_countdown_cap` countdown steps, and immunity to each
allele `evoepi_immunity_countdown_cap` steps, each step taken with
probability `evoepi_infection_countdown_p` or
`evoepi_immunity_countdown_p` per generation. Within-host evolution is
neutral (`evoepi_withinhost_b` 0), so with `evoepi_withinhost_t` 1, each
site of a new infection mutates with probability equal to the mutation
rate.

These runs' grids hold only PEs of full (not clipped) nested subgrids.

Nested subgrids: `nRowSubgrid` x `nRowSubgrid` subgrids at the corner
PE (row 0, col 0), then L-shaped shells of `factor`^2 - 1 subgrids,
each `factor` times larger, with subgrids overhanging the grid edge
clipped (wse-async-ga runtime config `nestedSubgridSizeFactor`).

For each run:
1. animates the whole grid, sampling every 9th PE along each axis, with
   nested subgrid boundaries overlaid as gray lines, transposed to
   landscape (PE rows as columns), keeping PE (0, 0) at top left;
2. animates every full (not clipped) subgrid, resized to the same panel
   size, with subgrid sizes in columns and replicate subgrids in rows;
   subgrids show every PE, up to `subgrid_max_res` PEs per side, with
   larger subgrids sampled evenly down to that resolution;
3. plots, in the same trellis, each subgrid's strain proportions over
   time as a stackplot, with and without uninfected hosts.

Colors: black, uninfected; white, wildtype; blue, site 0 mutant; red,
site 1 mutant; purple, double mutant.

In [ ]:
nbname = pathlib.Path(__file__).stem
outdir = pathlib.Path("outdata") / nbname
outdir.mkdir(parents=True, exist_ok=True)
outdir

PosixPath('outdata/2026-11-21-alleleevoepi-countdown-p2p-traitlog')

## Get Data

In [ ]:
# trait logger records, one OSF file per run (one job each)
# mutation rate 2^-16, 2^-20, 2^-24, 2^-28, 2^-32
osf_ids = ["u9kme", "h5cxv", "cq65h", "45jqf", "5ab3q"]

# wse-async-ga runtime config nestedSubgridSizeFactor of these runs,
# which trait logger output doesn't record
nest_factor = 2

# sample every nth PE along each axis for whole-grid animation; the
# smallest nested subgrid size (nRowSubgrid), so subgrid edges fall
# between sampled cells, and each sampled cell lies within one subgrid
grid_stride = 9

# most PEs per side to sample within each subgrid
subgrid_max_res = 72

config_columns = [
    "dispatchSlug",
    "globalSeed",
    "nRow",
    "nCol",
    "nRowSubgrid",
    "nColSubgrid",
    "tilePopSize",
    "tournSize",
    "nCycle",
    "p2pSendProbability",
    "evoepi_mutation_rate",
    "evoepi_infection_countdown_cap",
    "evoepi_infection_countdown_p",
    "evoepi_immunity_countdown_cap",
    "evoepi_immunity_countdown_p",
    "evoepi_withinhost_b",
    "evoepi_withinhost_t",
    "evoepi_seed_pe_stride",
    "dstream_storage_multiplex",
]

## Unpack Data

In [ ]:
def nested_subgrid_size(
    row: np.ndarray, col: np.ndarray, base: int, factor: int
) -> np.ndarray:
    """Size of each PE's nested subgrid."""
    shell = np.maximum(row, col) // base  # base-sized steps from corner
    size = np.full_like(shell, base)
    while (shell >= factor).any():
        grow = shell >= factor
        size[grow] *= factor
        shell[grow] //= factor
    return size

def sample_tiles(
    n_row: int,
    n_col: int,
    base: int,
    factor: int,
    grid_stride: int,
    max_res: int,
) -> pl.DataFrame:
    """PEs to unpack: whole-grid sample, and each full subgrid sampled
    at up to max_res x max_res resolution (subgrid_i, subgrid_j)."""
    rows, cols = np.indices((n_row, n_col))
    size = nested_subgrid_size(rows, cols, base, factor)
    top, left = rows - rows % size, cols - cols % size
    is_full = (top + size <= n_row) & (left + size <= n_col)
    stride = np.maximum(size // max_res, 1)
    is_subgrid_sample = (
        is_full & ((rows - top) % stride == 0) & ((cols - left) % stride == 0)
    )
    is_grid_sample = (rows % grid_stride == 0) & (cols % grid_stride == 0)
    keep = is_subgrid_sample | is_grid_sample
    return pl.DataFrame(
        {
            "row": rows[keep],
            "col": cols[keep],
            "is_grid_sample": is_grid_sample[keep],
            "is_subgrid_sample": is_subgrid_sample[keep],
            "subgrid_size": size[keep],
            "subgrid_top": top[keep],
            "subgrid_left": left[keep],
            "subgrid_res": (size // stride)[keep],
            "subgrid_i": ((rows - top) // stride)[keep],
            "subgrid_j": ((cols - left) // stride)[keep],
        },
    ).cast({"row": pl.UInt16, "col": pl.UInt16})

In [ ]:
def unpack_strains(df: pl.DataFrame) -> pl.DataFrame:
    """Strain time series of PEs in df, at their own logged times."""
    # unpack dstream data items from logger hex strings, demultiplexing
    # infected, site 0 allele, and site 1 allele records
    dfu = downstream.dataframe.unpack_data_packed(df)
    df = downstream.dataframe.explode_lookup_unpacked(
        dfu.with_columns(
            # fast forward loggers to at least dstream_S entries
            # (i.e., buffer filled)
            # these will be semantically-correct zero values
            dstream_T=pl.max_horizontal(
                pl.col("dstream_T"), pl.col("dstream_S")
            ),
        ),
        value_type="uint8",
    )

    # join metadata onto unpacked data
    df = df.join(
        dfu.select("dstream_data_id", "col", "row", "tile").unique(
            "dstream_data_id",
        ),
        on="dstream_data_id",
    )

    # apply dilation to dstream_Tbar, then one column per record
    df = (
        df.with_columns(
            dstream_Tbar=pl.col("dstream_Tbar")
            * pl.col("dstream_T_dilation"),
        )
        .pivot(
            on="dstream_storage_multiplex_id",
            index=["tile", "row", "col", "dstream_Tbar"],
            values="dstream_value",
        )
        .rename({"0": "infected", "1": "site0", "2": "site1"})
    )

    # addend time series groups with final uninfected entry, after
    # logger's last seen time (i.e., last time any record was set)
    df = addend_groups(
        df,
        group_by="tile",
        aggs=(
            pl.col("col").unique().pipe(squeeze_int),
            pl.col("row").unique().pipe(squeeze_int),
            (pl.col("dstream_Tbar") + 1).max().alias("dstream_Tbar"),
            pl.lit(0).alias("infected"),
            pl.lit(0).alias("site0"),
            pl.lit(0).alias("site1"),
        ),
        inner_only=True,
    )

    # 0: uninfected, 1: wildtype, 2: site 0 mutant, 3: site 1 mutant,
    # 4: double mutant
    return df.select(
        pl.col("tile").cast(pl.UInt32),
        pl.col("row").cast(pl.UInt16),
        pl.col("col").cast(pl.UInt16),
        pl.col("dstream_Tbar").cast(pl.UInt32),
        strain=(
            pl.col("infected") * (1 + pl.col("site0") + 2 * pl.col("site1"))
        ).cast(pl.UInt8),
    )

def unpack_strain_timeseries(
    df: pl.DataFrame, chunk_size: int = 2000
) -> tuple:
    """Strains of PEs in df, all at the union of their logged times.

    Returns times, PEs (tile, row, col), and strains, an array indexed
    by [time, PE]. Each PE's strain at a time is its strain at its last
    logged time at or before (or, if none, its first logged time).
    """
    # unpack in chunks of PEs, to save memory
    df = pl.concat(
        [
            unpack_strains(df.slice(start, chunk_size))
            for start in range(0, len(df), chunk_size)
        ],
    ).sort("tile", "dstream_Tbar")
    tiles = df.select("tile", "row", "col").unique("tile").sort("tile")
    times = np.sort(df["dstream_Tbar"].unique().to_numpy())

    # per PE, index of last logged time at or before each time (forward
    # fill), or first logged time if none (backward fill)
    tile_rank = np.searchsorted(tiles["tile"].to_numpy(), df["tile"].to_numpy())
    key = tile_rank.astype(np.int64) * (len(times) + 1) + np.searchsorted(
        times, df["dstream_Tbar"].to_numpy()
    )
    query = (
        np.arange(len(tiles), dtype=np.int64)[None, :] * (len(times) + 1)
        + np.arange(len(times), dtype=np.int64)[:, None]
    )
    first = np.searchsorted(tile_rank, np.arange(len(tiles)))
    index = np.maximum(np.searchsorted(key, query, side="right") - 1, first)
    strains = df["strain"].to_numpy()[index]
    assert strains.shape == (len(times), len(tiles))
    return times, tiles, strains

In [ ]:
# read and unpack one run at a time, keeping only sampled PEs' strains
_configs, runs = [], {}
for _osf_id in tqdm(osf_ids):
    _df = pl.read_parquet(
        f"https://osf.io/{_osf_id}/download",
        columns=[
            *config_columns,
            "tile",
            "row",
            "col",
            "data_hex",
            "dstream_algo",
            "dstream_storage_bitoffset",
            "dstream_storage_bitwidth",
            "dstream_S",
            "dstream_T_bitoffset",
            "dstream_T_bitwidth",
        ],
    ).with_row_index("dstream_data_id")
    _config = _df.select(config_columns).unique()
    assert len(_config) == 1
    _configs.append(_config.with_columns(osf_id=pl.lit(_osf_id)))
    _samples = sample_tiles(
        _config["nRow"].item(),
        _config["nCol"].item(),
        _config["nRowSubgrid"].item(),
        nest_factor,
        grid_stride,
        subgrid_max_res,
    )
    # keep only sampled PEs, and only columns needed to unpack
    _df = _df.join(_samples.select("row", "col"), on=["row", "col"]).select(
        "dstream_data_id",
        "tile",
        "row",
        "col",
        pl.col("^dstream_.*$").exclude("dstream_data_id"),
        "data_hex",
    )
    _times, _tiles, _strains = unpack_strain_timeseries(_df)
    del _df
    runs[_osf_id] = {
        "times": _times,
        "tiles": _tiles.join(_samples, on=["row", "col"], maintain_order="left"),
        "strains": _strains,
    }
    assert len(runs[_osf_id]["tiles"]) == _strains.shape[1]

configs = pl.concat(_configs)
assert len(configs) == len(osf_ids)
configs

  0%|          | 0/5 [00:00<?, ?it/s]/usr/local/lib/python3.10/dist-packages/downstream/dataframe/_impl/_check_downstream_version.py:34: UserWarning: Dataframe downstream_version column not provided
  warnings.warn(
100%|██████████| 5/5 [20:17<00:00, 243.52s/it]


dispatchSlug,globalSeed,nRow,nCol,nRowSubgrid,nColSubgrid,tilePopSize,tournSize,nCycle,p2pSendProbability,evoepi_mutation_rate,evoepi_infection_countdown_cap,evoepi_infection_countdown_p,evoepi_immunity_countdown_cap,evoepi_immunity_countdown_p,evoepi_withinhost_b,evoepi_withinhost_t,evoepi_seed_pe_stride,dstream_storage_multiplex,osf_id
cat,u32,u16,u16,u16,u16,u16,f32,u32,f32,f32,u8,f32,u8,f32,f32,f32,u16,u8,str
"""nested-mutlog2-16""",1131,1152,576,9,9,6000,2.0,50000,0.05,0.000015,10,1.0,15,0.03,0.0,1.0,9,3,"""u9kme"""
"""nested-mutlog2-20""",1133,1152,576,9,9,6000,2.0,50000,0.05,9.5367e-7,10,1.0,15,0.03,0.0,1.0,9,3,"""h5cxv"""
"""nested-mutlog2-24""",1135,1152,576,9,9,6000,2.0,50000,0.05,5.9605e-8,10,1.0,15,0.03,0.0,1.0,9,3,"""cq65h"""
"""nested-mutlog2-28""",1137,1152,576,9,9,6000,2.0,50000,0.05,3.7253e-9,10,1.0,15,0.03,0.0,1.0,9,3,"""45jqf"""
"""nested-mutlog2-32""",1139,1152,576,9,9,6000,2.0,50000,0.05,2.3283e-10,10,1.0,15,0.03,0.0,1.0,9,3,"""5ab3q"""


In [ ]:
pl.DataFrame(
    [
        {
            "osf_id": _osf_id,
            **{
                str(_k): int((_run["strains"] == _k).sum())
                for _k in range(5)
            },
        }
        for _osf_id, _run in runs.items()
    ],
)

osf_id,0,1,2,3,4
str,i64,i64,i64,i64,i64
"""u9kme""",112064523,1670588,1428893,1437493,1713511
"""h5cxv""",135588362,761238,2800526,2771724,585158
"""cq65h""",65239589,720193,0,0,165018
"""45jqf""",65298475,720783,0,0,105542
"""5ab3q""",65299230,719740,0,0,105830


## Animate Data

In [ ]:
palette = np.array(
    [
        [0, 0, 0],  # uninfected: black
        [255, 255, 255],  # wildtype: white
        [0, 0, 255],  # site 0 mutant: blue
        [255, 0, 0],  # site 1 mutant: red
        [160, 32, 240],  # double mutant: purple
    ],
    dtype=np.uint8,
)
strain_names = [
    "uninfected",
    "wildtype",
    "site 0 mutant",
    "site 1 mutant",
    "double mutant",
]

In [ ]:
def save_animation(outpath: str, frames: list) -> None:
    imageio.mimsave(f"{outpath}.gif", frames, duration=0.05, loop=0)
    # mp4 lasts 15 seconds, whatever its number of frames
    imageio.mimsave(
        f"{outpath}.mp4",
        frames,
        fps=len(frames) / 15,
        macro_block_size=1,
    )

def make_run_label(config: dict) -> str:
    return (
        f"mutlog2={round(np.log2(config['evoepi_mutation_rate']))}"
        f"+p2p={config['p2pSendProbability']:.3g}"
        f"+immunityp={config['evoepi_immunity_countdown_p']:.3g}"
        f"+seed={config['globalSeed']}"
    )

def subgrid_layout(tiles: pl.DataFrame) -> tuple:
    """Full subgrids, with replicate (row) and size (col) trellis
    positions, and each one's PE columns in strain arrays, row-major."""
    tiles = tiles.with_row_index("pe").filter("is_subgrid_sample")
    subgrids = (
        tiles.select(
            "subgrid_size", "subgrid_top", "subgrid_left", "subgrid_res"
        )
        .unique()
        .sort(pl.all())
        .with_columns(
            replicate=pl.int_range(pl.len()).over("subgrid_size"),
        )
    )
    sizes = sorted(subgrids["subgrid_size"].unique())
    num_reps = subgrids["replicate"].max() + 1
    layout = []
    for row in subgrids.iter_rows(named=True):
        pes = tiles.filter(
            (pl.col("subgrid_top") == row["subgrid_top"])
            & (pl.col("subgrid_left") == row["subgrid_left"])
        ).sort("subgrid_i", "subgrid_j")["pe"]
        assert len(pes) == row["subgrid_res"] ** 2
        layout.append(
            {
                **row,
                "size_index": sizes.index(row["subgrid_size"]),
                "pes": pes.to_numpy(),
            }
        )
    return layout, sizes, num_reps

In [ ]:
# whole grid, every grid_stride-th PE along each axis, with nested
# subgrid boundaries as gray lines
_scale = 4
for _osf_id, _run in tqdm(runs.items()):
    _config = configs.filter(pl.col("osf_id") == _osf_id).row(0, named=True)
    _label = make_run_label(_config)
    _tiles = (
        _run["tiles"]
        .with_row_index("pe")
        .filter("is_grid_sample")
        .sort("row", "col")
    )
    _num_rows = _tiles["row"].n_unique()
    _num_cols = _tiles["col"].n_unique()
    assert len(_tiles) == _num_rows * _num_cols
    _strains = _run["strains"][:, _tiles["pe"].to_numpy()]

    # gray lines at each nested subgrid's top and left edges, at their
    # exact PE positions, scaled to pixels
    _stride = grid_stride  # PEs per sampled cell
    _all_rows, _all_cols = np.indices((_config["nRow"], _config["nCol"]))
    _size = nested_subgrid_size(
        _all_rows, _all_cols, _config["nRowSubgrid"], nest_factor
    )
    _edges = np.unique(
        np.stack(
            [
                (_all_rows - _all_rows % _size).ravel(),
                (_all_cols - _all_cols % _size).ravel(),
                _size.ravel(),
            ],
        ),
        axis=1,
    )
    _boundary = np.zeros((_num_rows * _scale, _num_cols * _scale), dtype=bool)
    _px = lambda _pe: int(round(_pe / _stride * _scale))
    for _top, _left, _sz in _edges.T:
        _y0, _x0 = _px(_top), _px(_left)
        _y1 = min(_px(_top + _sz), _boundary.shape[0])
        _x1 = min(_px(_left + _sz), _boundary.shape[1])
        if _top > 0 and _y0 < _boundary.shape[0]:
            _boundary[_y0, _x0:_x1] = True
        if _left > 0 and _x0 < _boundary.shape[1]:
            _boundary[_y0:_y1, _x0] = True

    _frames = []
    for _s in _strains:
        _frame = np.kron(
            palette[_s.reshape(_num_rows, _num_cols)],
            np.ones((_scale, _scale, 1), dtype=np.uint8),
        )
        _frame[_boundary] = 128
        # landscape: transpose, keeping PE (0, 0) at top left
        _frames.append(np.ascontiguousarray(_frame.transpose(1, 0, 2)))
    print(_osf_id, _label, len(_frames), _frames[0].shape)
    save_animation(
        outdir / f"a=strains-grid+{_label}+osf={_osf_id}+ext=", _frames
    )

  0%|          | 0/5 [00:00<?, ?it/s]

u9kme mutlog2=-16+p2p=0.05+immunityp=0.03+seed=1131 1834 (256, 512, 3)


 20%|██        | 1/5 [00:14<00:58, 14.53s/it]

h5cxv mutlog2=-20+p2p=0.05+immunityp=0.03+seed=1133 2209 (256, 512, 3)


 40%|████      | 2/5 [00:30<00:45, 15.18s/it]

cq65h mutlog2=-24+p2p=0.05+immunityp=0.03+seed=1135 1025 (256, 512, 3)


 60%|██████    | 3/5 [00:36<00:22, 11.33s/it]

45jqf mutlog2=-28+p2p=0.05+immunityp=0.03+seed=1137 1025 (256, 512, 3)


 80%|████████  | 4/5 [00:43<00:09,  9.49s/it]

5ab3q mutlog2=-32+p2p=0.05+immunityp=0.03+seed=1139 1025 (256, 512, 3)


100%|██████████| 5/5 [00:50<00:00, 10.03s/it]


In [ ]:
# every full subgrid, resized to same panel size; sizes in cols,
# replicates in rows
_panel_px, _gap, _label_height = 2 * subgrid_max_res, 6, 28
_font = ImageFont.load_default(size=16)
for _osf_id, _run in tqdm(runs.items()):
    _config = configs.filter(pl.col("osf_id") == _osf_id).row(0, named=True)
    _label = make_run_label(_config)
    _layout, _sizes, _num_reps = subgrid_layout(_run["tiles"])

    # layout: label strip, then subgrid panels, gray gaps between
    _height = _label_height + _num_reps * (_panel_px + _gap) + _gap
    _width = len(_sizes) * (_panel_px + _gap) + _gap
    _height += _height % 2
    _width += _width % 2
    _background = Image.new("RGB", (_width, _height), (128, 128, 128))
    _draw = ImageDraw.Draw(_background)
    for _k, _size in enumerate(_sizes):  # column labels
        _text = f"{_size} x {_size}"
        _tw = _draw.textlength(_text, font=_font)
        _draw.text(
            (_gap + _k * (_panel_px + _gap) + (_panel_px - _tw) / 2, 6),
            _text,
            font=_font,
            fill=(255, 255, 255),
        )
    _background = np.asarray(_background)

    _frames = []
    for _s in _run["strains"]:
        _frame = _background.copy()
        for _sub in _layout:
            _res = _sub["subgrid_res"]
            _scale = _panel_px // _res
            _y = _label_height + _gap + _sub["replicate"] * (_panel_px + _gap)
            _x = _gap + _sub["size_index"] * (_panel_px + _gap)
            _frame[
                _y : _y + _res * _scale, _x : _x + _res * _scale
            ] = np.kron(
                palette[_s[_sub["pes"]].reshape(_res, _res)],
                np.ones((_scale, _scale, 1), dtype=np.uint8),
            )
        _frames.append(_frame)
    print(_osf_id, _label, len(_frames), _frames[0].shape)
    save_animation(
        outdir / f"a=strains-subgrids+{_label}+osf={_osf_id}+ext=", _frames
    )

  0%|          | 0/5 [00:00<?, ?it/s]

u9kme mutlog2=-16+p2p=0.05+immunityp=0.03+seed=1131 1834 (634, 1056, 3)


 20%|██        | 1/5 [01:17<05:09, 77.26s/it]

h5cxv mutlog2=-20+p2p=0.05+immunityp=0.03+seed=1133 2209 (634, 1056, 3)


 40%|████      | 2/5 [02:51<04:21, 87.18s/it]

cq65h mutlog2=-24+p2p=0.05+immunityp=0.03+seed=1135 1025 (634, 1056, 3)


 60%|██████    | 3/5 [03:25<02:05, 62.81s/it]

45jqf mutlog2=-28+p2p=0.05+immunityp=0.03+seed=1137 1025 (634, 1056, 3)


 80%|████████  | 4/5 [03:58<00:51, 51.25s/it]

5ab3q mutlog2=-32+p2p=0.05+immunityp=0.03+seed=1139 1025 (634, 1056, 3)


100%|██████████| 5/5 [04:32<00:00, 54.48s/it]


## Plot Data

In [ ]:
# each full subgrid's strain proportions over time, in same trellis as
# subgrid animation; with uninfected hosts (black), and among infected
# hosts only
def _plot_stacks(axes, layout, times, strains, with_uninfected):
    strain_order = [1, 2, 3, 4] + ([0] if with_uninfected else [])
    for _ax in axes.flat:
        _ax.set_visible(False)
    for _sub in layout:
        _ax = axes[_sub["replicate"], _sub["size_index"]]
        _ax.set_visible(True)
        _counts = np.stack(
            [(strains[:, _sub["pes"]] == _k).sum(axis=1) for _k in strain_order]
        ).astype(float)
        _total = _counts.sum(axis=0)
        _props = np.divide(
            _counts, _total, out=np.zeros_like(_counts), where=_total > 0
        )
        _ax.stackplot(
            times,
            _props,
            colors=[palette[_k] / 255 for _k in strain_order],
            labels=[strain_names[_k] for _k in strain_order],
            edgecolor="none",
            linewidth=0.2,
        )
        _ax.set_ylim(0, 1)
        _ax.set_xlim(times.min(), times.max())
        _ax.set_facecolor("lightgray")
        if _sub["replicate"] == 0:
            _ax.set_title(f"{_sub['subgrid_size']} x {_sub['subgrid_size']}")
    _ax = axes[0, 0]
    _ax.figure.legend(*_ax.get_legend_handles_labels())
    sns.move_legend(
        _ax.figure,
        "lower center",
        bbox_to_anchor=(0.5, 1.0),
        ncol=len(strain_order),
        frameon=False,
        title=None,
    )
    _ax.figure.supxlabel("Generations")
    _ax.figure.supylabel(
        "Proportion of Hosts" if with_uninfected
        else "Proportion of Infected Hosts"
    )

for _osf_id, _run in runs.items():
    _config = configs.filter(pl.col("osf_id") == _osf_id).row(0, named=True)
    _layout, _sizes, _num_reps = subgrid_layout(_run["tiles"])
    for _with_uninfected in (True, False):
        with tp.teed(
            plt.subplots,
            nrows=_num_reps,
            ncols=len(_sizes),
            figsize=(2 * len(_sizes), 1.6 * _num_reps),
            sharex=True,
            sharey=True,
            squeeze=False,
            constrained_layout=True,
            teeplot_outattrs={
                "a": "strain-stacks",
                "osf": _osf_id,
                "uninfected": _with_uninfected,
                **dict(
                    _kv.split("=")
                    for _kv in make_run_label(_config).split("+")
                ),
            },
            teeplot_subdir=nbname,
        ) as (_fig, _axes):
            _plot_stacks(
                _axes,
                _layout,
                _run["times"],
                _run["strains"],
                _with_uninfected,
            )

teeplots/2026-11-21-alleleevoepi-countdown-p2p-traitlog/a=strain-stacks+immunityp=0.03+mutlog2=-16+osf=u9kme+p2p=0.05+seed=1131+uninfected=True+viz=subplots+ext=.pdf
teeplots/2026-11-21-alleleevoepi-countdown-p2p-traitlog/a=strain-stacks+immunityp=0.03+mutlog2=-16+osf=u9kme+p2p=0.05+seed=1131+uninfected=True+viz=subplots+ext=.png
teeplots/2026-11-21-alleleevoepi-countdown-p2p-traitlog/a=strain-stacks+immunityp=0.03+mutlog2=-16+osf=u9kme+p2p=0.05+seed=1131+uninfected=False+viz=subplots+ext=.pdf
teeplots/2026-11-21-alleleevoepi-countdown-p2p-traitlog/a=strain-stacks+immunityp=0.03+mutlog2=-16+osf=u9kme+p2p=0.05+seed=1131+uninfected=False+viz=subplots+ext=.png
teeplots/2026-11-21-alleleevoepi-countdown-p2p-traitlog/a=strain-stacks+immunityp=0.03+mutlog2=-20+osf=h5cxv+p2p=0.05+seed=1133+uninfected=True+viz=subplots+ext=.pdf
teeplots/2026-11-21-alleleevoepi-countdown-p2p-traitlog/a=strain-stacks+immunityp=0.03+mutlog2=-20+osf=h5cxv+p2p=0.05+seed=1133+uninfected=True+viz=subplots+ext=.png
te